# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook demonstrates step-by-step exploration and processing of the FAIR² colorectal second-primary cancer patient dataset, using the [`mlcroissant`](https://mlcroissant.readthedocs.io/) library and consistently referencing all dataset entities by their `@id` as required by the Croissant and MLCommons conventions.

### Dataset Source

The dataset is described by a Croissant schema at:

```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```
This schema exposes metadata, record sets, fields, and columns by their Croissant `@id`s.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
We will load the dataset metadata and record structure using the `mlcroissant` API. All references to record sets, fields, and columns use their unique `@id` fields.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the Croissant dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Print dataset high-level info
print(f"Dataset name: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Identifier: {metadata.identifier}")
print(f"Version: {metadata.version}")

## 2. Data Overview

List available record sets (tables), their field `@id`s, and column `@id`s. This step uses the Croissant metadata and dataset API and is useful for identifying how to access and reference all data elements.

In [ ]:
# List all record sets and their @id's
print("Available Record Sets (by @id):")
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_sets = metadata.recordSet
    if not isinstance(record_sets, list):
        record_sets = [record_sets]
    for rs in record_sets:
        print(f"  - {rs['@id']}")
else:
    # Use dataset.record_sets for full compatibility
    record_sets = dataset.record_sets
    for rs in record_sets:
        print(f"  - {rs['@id']}")

if not record_sets:
    print("No record sets found via metadata; inferring record sets from dataset.record_sets API...")
    record_sets = dataset.record_sets

# Show fields for each record set (by @id)
for rs in record_sets:
    print(f"\nRecord Set: {rs['@id']}")
    if 'field' in rs:
        fields = rs['field']
        if not isinstance(fields, list):
            fields = [fields]
        for f in fields:
            # Each field is a dict; Croissant standard: use @id and name
            print(f"    Field: {f['@id']} (name: {f.get('name', '')})")
            if 'column' in f:
                columns = f['column']
                if not isinstance(columns, list):
                    columns = [columns]
                for col in columns:
                    print(f"        Column: {col['@id']} (name: {col.get('name', '')})")

## 3. Data Extraction

Load the actual records from the primary record set(s) using the `@id`, and build pandas DataFrames for further analysis. Always use the `@id` reference to select record sets and fields for extraction.

**Note:** Some Croissant datasets only expose a single main record set; others may have several.

In [ ]:
# Gather all record set @id's
record_set_ids = [rs['@id'] for rs in dataset.record_sets]

print("\nExtracting data from these record sets:")
for rid in record_set_ids:
    print(f"  - {rid}")

dataframes = {}
for record_set_id in record_set_ids:
    print(f"\nLoading DataFrame for Record Set '{record_set_id}' ...")
    # Download all records for this record set (@id)
    records_iter = dataset.records(record_set=record_set_id)
    records = list(records_iter)
    if not records:
        print(f"  No records found for {record_set_id}")
        continue
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"  Loaded columns (by field @id): {df.columns.tolist()}")
    # Preview
    display(df.head())

## 4. Exploratory Data Analysis (EDA)

Let's perform some basic data cleaning, normalization, and grouping on the main record set. All column/field accesses use the field or column `@id` as previously discovered.

**You can easily adapt this section for your own columns and numeric fields!**

In [ ]:
# We will pick the first available record set for demonstration:
main_record_set_id = record_set_ids[0] if record_set_ids else None
if not main_record_set_id:
    raise ValueError("No record sets found in this dataset.")
main_df = dataframes[main_record_set_id]

# Display auto-detected numeric columns by checking dtype
numeric_cols = main_df.select_dtypes(include=['int64', 'float64']).columns.tolist()
print(f"Numeric columns (@id): {numeric_cols}")

# If there are none auto-detected, try to infer from field names (e.g. by heuristic)
if not numeric_cols:
    numeric_guess = [col for col in main_df.columns if any(sub in col.lower() for sub in ["age", "interval", "score", "count"]) ]
    numeric_cols = numeric_guess

if numeric_cols:
    numeric_field_id = numeric_cols[0]  # Use first numeric field
    print(f"Using {numeric_field_id} as the numeric field for demonstration.")

    # Outlier filtering
    threshold = main_df[numeric_field_id].mean()  # Use mean as example threshold
    filtered_df = main_df[main_df[numeric_field_id] > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    display(filtered_df.head())

    # Normalize
    norm_col = f"{numeric_field_id}_normalized"
    filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    display(filtered_df[[numeric_field_id, norm_col]].head())

    # Group by a likely categorical field — heuristically pick the first string/dtype object column
    obj_cols = main_df.select_dtypes(include=['object']).columns.tolist()
    # Try to avoid grouping by trivial columns like '@id', so filter by length/name
    group_field = next((col for col in obj_cols if len(main_df[col].unique()) < 20 and not col.startswith('@id')), obj_cols[0]) if obj_cols else None
    if group_field:
        grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean().reset_index()
        print(f"Grouped data: mean {numeric_field_id} by {group_field}")
        display(grouped_df)
else:
    print("No numeric columns found for EDA. Please check your data or review available field @ids.")

## 5. Visualization
Create simple plots using pandas/matplotlib for the selected numeric field and a grouping categorical field. All axes should be labeled with the field's @id for reproducibility with Croissant metadata.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_cols:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[numeric_field_id].dropna(), bins=15)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If we have a group_field
    if 'group_field' in locals() and group_field:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=main_df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field_id)
        plt.show()
else:
    print("No numeric columns available for visualization.")

## 6. Conclusion

In this notebook, we loaded and explored the FAIR² second primary colorectal cancer dataset, listed all data structures by their Croissant `@id`, extracted tabular records, and performed example data processing and visualization by referencing all columns by their IDs. This reproducible workflow ensures all steps are traceable to standard-compliant schema elements, and you can adapt this approach for other Croissant datasets or advanced analysis!